# Cross integration: Multiple RNA + ADT + ATAC

Cross integration combines batches that all measure the same modalities. The task is to remove batch effects and keep the biological structure. This tutorial is for batches that each hold RNA, ADT and ATAC peaks. It runs StabMap and UINMF on `D59mini`, 3,000 cells in two batches of the benchmark dataset `D59`, and then on data in your own format.

Methods run on Linux. On macOS or Windows, [open this notebook in Colab](https://colab.research.google.com/github/DSichang/scMultiBench/blob/main/notebooks/tutorial_cross_rna_adt_atac.ipynb).

## 1. Install

This cell installs `multibench-sc`. On Colab, choose a T4 GPU runtime first (Runtime -> Change runtime type), because changing it later loses the downloads.

In [ ]:
import numpy, pandas
pins = f"numpy=={numpy.__version__} pandas=={pandas.__version__}"
%pip install -q "multibench-sc>=0.5.0" {pins}

## 2. Download the data and the environments

`mtb.data.fetch` downloads `D59mini` (8 MB) once. `mtb.env.install` downloads the environments for StabMap and UINMF. The download is 0.9 GB.

In [ ]:
import multibench as mtb

mtb.data.fetch("D59mini")
METHODS = ["StabMap", "UINMF"]
envs = mtb.env.install(METHODS)

## 3. Run the methods

`run` runs each method on `D59mini` and scores its output with the scIB metrics. The clustering metrics measure how well the embedding separates the cell types. The batch metrics measure how well the batches mix. The classification scores measure how well the cell types learned on one batch are predicted in the other batches. `res.summary` has one row per method with its status, run time and scores. Higher is better for every metric.

<details>
<summary>Details</summary>

`params={"Method": {"key": value}}` sets a method's parameters, and `mtb.params_for` lists them. Many methods take none.

`mtb.load_run("out/D59mini")` reloads these results later without running anything.

</details>

In [ ]:
res = mtb.run("D59mini", "cross", methods=METHODS,
                  out_dir="out/D59mini")
res.summary

## 4. Plot

`res.plot()` draws the scores as a bubble table. The lightest fill is the lowest value in this figure, not zero.

<details>
<summary>Details</summary>

Metrics are grouped by family: blue for clustering, green for batch correction, pink for classification. Each family starts with an `Overall` bar. Its length and colour both show the family score.

A column whose rows all hold the same value is drawn grey, and a warning names it. A figure of one method is grey everywhere.

</details>

In [ ]:
res.plot()

## 5. Your own data

Your data needs raw RNA and ADT counts in one `.h5ad` file and an ATAC peak matrix of the same cells in another, with a cell type and a batch for each cell. A batch is a donor, a sample or an experiment. `mtb.io.export_dataset` with `batch=` writes one set of files per batch.

The methods of this task read two batches.

Starting from something else? See the recipes for [a 10x `.h5` file](https://dsichang.github.io/scMultiBench/tutorials/run/#recipe-10x-h5), [labels in a CSV file](https://dsichang.github.io/scMultiBench/tutorials/run/#recipe-labels-csv), [no labels](https://dsichang.github.io/scMultiBench/tutorials/run/#recipe-no-labels) and [another number of donors](https://dsichang.github.io/scMultiBench/tutorials/run/#recipe-donors).

<details>
<summary>Details: export</summary>

The names in quotes say where your file holds each part. `"X"` is the main matrix, `"obsm:protein"` the matrix `obsm["protein"]`, `"layer:counts"` the layer `layers["counts"]`, and `"obs:celltype"` the column `obs["celltype"]`, which `"celltype"` alone names too. A wrong name after `obsm:`, `layer:` or `obs:` raises an error that lists the names the file has.

The example keeps the protein names in `adata.uns["protein_names"]`. A DataFrame in `adata.obsm["protein"]` brings its own names in its columns.

The batches are numbered in the sorted order of their names, read as text: donors `A`, `B` and `C` become batches 1, 2 and 3, and `10` comes before `2`.

For one AnnData per batch, call `export_dataset` once per batch with `batch_index=N` instead of `batch=`.

Without `methods=`, `scan` lists every method that reads the modalities in the folder.

</details>

The cells below read your data, write it as a dataset folder, check the folder and run the same methods on it. So that the notebook runs as it is, the next cell writes two small example files, `my_file.h5ad` and `my_atac.h5ad`, from 60% of the cells of `D59mini`, with a batch column.

In [ ]:
mtb.data.write_example("D59mini")

Read your files. For your own data, change every file name marked `# <-`. On Colab, upload the files first: click the folder icon on the left, then the upload icon.

In [ ]:
import anndata as ad

adata = ad.read_h5ad("my_file.h5ad")   # <- your RNA and protein file
atac = ad.read_h5ad("my_atac.h5ad")    # <- your peak file, the same cells
print(adata)
print(atac)

Write the dataset folder. Change each name in quotes marked `# <-`: give `NAME` a new value, and write the names that your files use for the other parts. The printout above lists the names in your files. Write a name listed after `obs:` with that prefix, as `"obs:celltype"`, and one after `obsm:` as `"obsm:protein"`.

Compare the cell counts above the table with your data.

In [ ]:
NAME = "MYCROSS_TRI"   # <- a new name for your own data
mtb.io.export_dataset(adata, "mydata/" + NAME,
                      rna="X",                # <- where the RNA counts are:
                                              #    "X" or "layer:counts"
                      adt="obsm:protein",     # <- where the proteins (ADT) are
                      atac=atac,              # the ATAC file, peak counts in X
                      atac_kind="peak",       # the methods here read peaks
                      labels="obs:celltype",  # <- the cell-type column
                      batch="obs:batch",      # <- the batch or donor column
                      category="cross", overwrite=True)
mtb.scan("mydata/" + NAME, "cross", methods=METHODS)

Run the same methods on the folder:

In [ ]:
mine = mtb.run("mydata/" + NAME, "cross", methods=METHODS,
                   out_dir="out/" + NAME)
mine.summary

In [ ]:
mine.plot()

## 6. Every method of this task

5 methods have a variant for this task. [This notebook](https://colab.research.google.com/github/DSichang/scMultiBench/blob/main/notebooks/tutorial_cross_rna_adt_atac_all.ipynb) runs them all. It downloads 4 envs, 15.8 GB on a CPU host, 15.5 GB on a GPU host.

## Troubleshooting

`res.summary[res.summary.status != "scored"]` lists each method that failed or was skipped. Its `reason` column ends with the method's error output.

<details>
<summary>Details</summary>

| symptom | fix |
|---|---|
| a method fails on a small dataset | `mtb.scan` names what the method needs from the data's size in its `caveat` column |
| a warning that values are not whole numbers | export raw counts, for example with `rna="layer:counts"` |
| `... matrix/data as cells x features` | the matrix is transposed: export it again with `mtb.io.export_dataset` |
| a method times out | raise `timeout=` in `run` |
| batch metrics use the wrong batches | `res.rescore(batch=my_vector)` scores again without running the methods |

</details>

## Next steps

- The guides: [run](https://dsichang.github.io/scMultiBench/tutorials/run/), [evaluate](https://dsichang.github.io/scMultiBench/tutorials/evaluate/), [plot](https://dsichang.github.io/scMultiBench/tutorials/plot/) and [discover methods](https://dsichang.github.io/scMultiBench/tutorials/discover/).
- The [interactive explorer](https://shiny.maths.usyd.edu.au/scMultiBench/) has the full benchmark's rankings, with no install needed.